In [3]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sentence_transformer,
    load_triplets,
)


In [5]:
import os
import shutil
from pathlib import Path
from sentence_transformers import SentenceTransformer
from peft import PeftModel

# ---- required eval utils must already be imported in notebook ----
# from utils import load_triplets, build_global_corpus, build_eval_items, build_binary_relevance_maps, evaluate_sentence_transformer

BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"

DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_dbpedia.json")
TOPKS = [3, 5, 10, 20]
BATCH = 32

# 1) Build eval objects ONCE
raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")

# 2) Epoch checkpoints
CKPT_ROOT = Path(PROJECT_ROOT) / "../finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/checkpoints"
TMP_ROOT  = Path(PROJECT_ROOT) / "../finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/tmp_merged_eval"
TMP_ROOT.mkdir(parents=True, exist_ok=True)

steps_per_epoch = 7869
epochs = [1, 2, 3, 4, 5]

for epoch in epochs:
    step = epoch * steps_per_epoch
    adapter_dir = (CKPT_ROOT / f"checkpoint-{step}").resolve()
    merged_dir  = (TMP_ROOT / f"epoch_{epoch}_merged").resolve()

    if not adapter_dir.exists():
        print(f"\n[skip] epoch {epoch}: checkpoint not found -> {adapter_dir}")
        continue

    print(f"\n=== Epoch {epoch} | adapter: {adapter_dir} ===")

    # load base -> attach LoRA adapter -> merge
    model = SentenceTransformer(BASE_MODEL)
    word_module = model[0]
    peft_model = PeftModel.from_pretrained(word_module.auto_model, str(adapter_dir))
    word_module.auto_model = peft_model.merge_and_unload()

    # save merged temp model
    if merged_dir.exists():
        shutil.rmtree(merged_dir)
    model.save(str(merged_dir))

    # evaluate
    evaluate_sentence_transformer(
        model_name=str(merged_dir),
        display_name=f"Fine-tuned: Qwen DBpedia (epoch {epoch})",
        eval_items=eval_items,
        global_corpus=global_corpus,
        relevance_scores_per_item=relevance_scores_per_item,
        topks=TOPKS,
        batch_size=BATCH,
    )

Loaded 337 items
Global corpus size: 1677
Eval anchors with non-empty positives: 337

=== Epoch 1 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/checkpoints/checkpoint-7869 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5643.84it/s]



Fine-tuned: Qwen DBpedia (epoch 1)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0465  0.0144  0.0203  0.0678  0.1335    0.0465  0.0411
@5     0.0380  0.0194  0.0234  0.0759  0.1691    0.0380  0.0374
@10    0.0341  0.0306  0.0292  0.0826  0.2226    0.0341  0.0382
@20    0.0329  0.0628  0.0386  0.0893  0.3205    0.0329  0.0500

=== Epoch 2 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/checkpoints/checkpoint-15738 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6830.79it/s]



Fine-tuned: Qwen DBpedia (epoch 2)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0218  0.0041  0.0068  0.0396  0.0564    0.0218  0.0228
@5     0.0243  0.0082  0.0118  0.0470  0.0890    0.0243  0.0244
@10    0.0255  0.0238  0.0216  0.0577  0.1721    0.0255  0.0274
@20    0.0270  0.0475  0.0303  0.0634  0.2611    0.0270  0.0356

=== Epoch 3 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/checkpoints/checkpoint-23607 ===


...


Fine-tuned: Qwen DBpedia (epoch 3)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0406  0.0109  0.0156  0.0584  0.0950    0.0406  0.0392
@5     0.0362  0.0155  0.0196  0.0649  0.1246    0.0362  0.0374
@10    0.0347  0.0279  0.0277  0.0736  0.1929    0.0347  0.0385
@20    0.0312  0.0500  0.0348  0.0784  0.2641    0.0312  0.0443

=== Epoch 4 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/checkpoints/checkpoint-31476 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6487.74it/s]



Fine-tuned: Qwen DBpedia (epoch 4)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0534  0.0144  0.0211  0.0984  0.1365    0.0534  0.0569
@5     0.0510  0.0234  0.0294  0.1103  0.1899    0.0510  0.0550
@10    0.0445  0.0405  0.0382  0.1173  0.2433    0.0445  0.0531
@20    0.0412  0.0770  0.0482  0.1239  0.3323    0.0412  0.0654

=== Epoch 5 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/checkpoints/checkpoint-39345 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5276.54it/s]



Fine-tuned: Qwen DBpedia (epoch 5)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0227  0.0040  0.0065  0.0420  0.0534    0.0227  0.0251
@5     0.0237  0.0073  0.0106  0.0472  0.0772    0.0237  0.0251
@10    0.0249  0.0170  0.0185  0.0551  0.1365    0.0249  0.0264
@20    0.0289  0.0472  0.0323  0.0625  0.2404    0.0289  0.0367
